# 05 — pandas for AI Engineers

    - Inspect and clean tabular data
- Filter, aggregate, and join
- Avoid preprocessing leakage

    **Estimated study/practice time:** 3–4 hours  
    **Prerequisites:** Notebook 04; basic tabular concepts

    **Priority legend:** 🔴 **Must Know** · 🟠 **Important** · 🟢 **Good to Know**

    ## Table of contents

    1. [DataFrame basics](#dataframe-basics)
2. [Selection and filtering](#selection-and-filtering)
3. [Missing data](#missing-data)
4. [GroupBy and merge](#groupby-and-merge)
5. [Vectorization and features](#vectorization-and-features)
6. [20 exercises](#20-exercises)
7. [ML mini-project](#ml-mini-project)

    > Active-practice rule: run examples, predict output before revealing it, complete every `TODO`, then compare with the solution below it.


## DataFrame basics

🔴 **Must Know** — **Concept:** a DataFrame is a labeled 2D table. **Why it matters:** AI interviews test rapid inspection and safe preprocessing. **Syntax:** `head`, `shape`, `columns`, `dtypes`, `info`, `describe`. **How pandas executes it:** columns are aligned labeled arrays that may use different dtypes. **Complexity:** metadata access is small/O(1); summaries scan relevant values. **Common mistake:** transforming before inspecting schema and missingness. **Interview question:** Series versus DataFrame? One labeled dimension versus two. **Exercise/solution:** complete *select features* and *missing counts*.


In [ ]:
import numpy as np
import pandas as pd
df = pd.DataFrame({
    "user_id": [1,2,3,4,4], "age": [25,np.nan,40,35,35],
    "country": ["SG","US","SG","MY","MY"], "salary": [60,80,100,70,70],
    "label": [0,1,1,0,0], "date": ["2026-01-01","2026-01-02","2026-02-01","2026-02-02","2026-02-02"]
})
print(df.head(), df.shape, list(df.columns), df.dtypes)
df.info()
print(df.select_dtypes("number").describe())


## Selection and filtering

🔴 **Must Know** — **Concept:** column selection chooses labels; `loc` uses labels and `iloc` positions; masks filter rows. **Why it matters:** feature selection and cohort analysis are everyday ML tasks. **Syntax:** `df['age']`, `df[['age','salary']]`, `.loc`, `.iloc`, and parenthesized masks. **How pandas executes it:** labels align operations; a boolean Series chooses matching rows. **Complexity:** filtering scans O(n) rows and produces output-sized storage. **Common mistake:** omitting parentheses around comparisons joined by `&`/`|`. **Interview question:** `loc` versus `iloc`? Label-based/inclusive label slices versus position-based/exclusive positional slices. **Exercise/solution:** complete *filter adults*, *SG high salary*, and *query IDs*.


In [ ]:
print(df["age"])
print(df[["age", "salary"]])
print(df.loc[df["country"].eq("SG"), ["user_id", "salary"]])
print(df.iloc[:2, :3])
subset = df[(df["age"] > 30) & (df["country"] == "SG")]
print(subset)


## Missing data

🔴 **Must Know** — **Concept:** missing values require explicit detection and a domain-appropriate policy. **Why it matters:** careless imputation can bias models or leak evaluation data. **Syntax:** `isna`, `dropna`, `fillna`; mean/median/mode/domain sentinel. **How pandas executes it:** masks identify missing markers and methods return transformed frames unless `inplace` is requested. **Complexity:** detection/filling scans O(nc) cells. **Common mistake:** fitting imputation statistics on validation/test data. **Interview question:** why use a training median? Robustness and leakage prevention. **Exercise/solution:** complete *fill age*.


In [ ]:
print(df.isna().sum())
train_age_median = df["age"].median()
filled = df.assign(age=df["age"].fillna(train_age_median))
print(filled.dropna(), train_age_median)


## GroupBy and merge

🔴 **Must Know** — **Concept:** GroupBy is split → apply → combine; merge is SQL-style key alignment. **Why it matters:** ML datasets often combine metadata and predictions then summarize cohorts. **Syntax:** `groupby(...).agg(...)` and `pd.merge(..., how='left')`. **How pandas executes it:** grouping hashes/sorts keys; merge builds key matches and can multiply rows. **Complexity:** usually O(n) average grouping and O(n+m) average hash joins, with output-dependent space. **Common mistake:** ignoring duplicate-key cardinality; use `validate=`. **Interview question:** what does a left join preserve? Every left row. **Exercise/solution:** complete *country summary* and *left join predictions*.

```text
LEFT JOIN on user_id
users:       1 A      predictions: 1 .9      result: 1 A .9
             2 B                   3 .7              2 B NaN
```


In [ ]:
summary = df.groupby("country").agg(mean_salary=("salary","mean"), users=("salary","count"))
predictions = pd.DataFrame({"user_id":[1,3], "score":[.9,.7]})
joined = pd.merge(df, predictions, on="user_id", how="left", validate="many_to_one")
print(summary, joined)


## Vectorization and features

🟠 **Important** — **Concept:** vectorized column, string, and datetime operations transform whole Series. **Why it matters:** efficient feature engineering includes class balance, duplicate control, and temporal features. **Syntax:** arithmetic columns, `value_counts`, `drop_duplicates`, `pd.to_datetime`, `.dt`. **How pandas executes it:** vectorized kernels operate over arrays; `apply` invokes a callable per element/row and remains appropriate for irreducible custom logic. **Complexity:** most transformations scan O(n); sorting/deduplication may add hashing or O(n log n) sorting. **Common mistake:** saying `apply` is always bad or parsing dates after using `.dt`. **Interview question:** why inspect duplicates before splitting? The same record can leak across train/test. **Exercise/solution:** complete *label distribution*, *unique users*, and *parse month*.


In [ ]:
featured = df.assign(
    salary_adjusted=df["salary"] * 1.1,
    date=pd.to_datetime(df["date"]),
).drop_duplicates()
featured["year"] = featured["date"].dt.year
featured["month"] = featured["date"].dt.month
featured["dayofweek"] = featured["date"].dt.dayofweek
print(df["label"].value_counts(), df["label"].value_counts(normalize=True))
print(featured.sort_values("salary", ascending=False))


## 20 exercises


### Exercise — 🟢 Easy: 1. select_features

        **Goal:** Return age and salary columns.

        **Example**

        ```text
        Input columns: age, salary, label
Output columns: age, salary
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def select_features(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def select_features(df):
    return df[['age','salary']]

assert list(select_features(pd.DataFrame({'age':[1],'salary':[2]})).columns)==['age','salary']


**Complexity**

```text
Time: O(n); Space: O(n) for the returned frame
```

**Common mistake:** Using one bracket returns a Series when a DataFrame is required.


### Exercise — 🟢 Easy: 2. filter_adults

        **Goal:** Keep age >= 18.

        **Example**

        ```text
        Input ages: [17,18]
Output: row with age 18
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def filter_adults(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def filter_adults(df):
    return df.loc[df['age'] >= 18].copy()

assert len(filter_adults(pd.DataFrame({'age':[17,18]})))==1


**Complexity**

```text
Time: O(n); Space: O(n) output
```

**Common mistake:** Using `>` accidentally excludes age 18.


### Exercise — 🟢 Easy: 3. sg_high_salary

        **Goal:** Keep SG rows with salary > threshold.

        **Example**

        ```text
        Input: SG/10 and US/20, threshold=5
Output: SG/10 row
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def sg_high_salary(df, threshold):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def sg_high_salary(df, threshold):
    return df.loc[(df['country']=='SG') & (df['salary']>threshold)].copy()

assert len(sg_high_salary(pd.DataFrame({'country':['SG','US'],'salary':[10,20]}),5))==1


**Complexity**

```text
Time: O(n); Space: O(n) output
```

**Common mistake:** Omitting parentheses around each boolean condition.


### Exercise — 🟢 Easy: 4. missing_counts

        **Goal:** Count missing values per column.

        **Example**

        ```text
        Input column x=[1,NaN]
Output: x=1
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def missing_counts(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def missing_counts(df):
    return df.isna().sum()

assert missing_counts(pd.DataFrame({'x':[1,np.nan]}))['x']==1


**Complexity**

```text
Time: O(nc); Space: O(c)
```

**Common mistake:** Calling `sum()` before `isna()`.


### Exercise — 🟢 Easy: 5. fill_age

        **Goal:** Fill age with supplied training median.

        **Example**

        ```text
        Input age=[NaN], training median=20
Output age=[20]
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def fill_age(df, train_median):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def fill_age(df, train_median):
    return df.assign(age=df['age'].fillna(train_median))

assert fill_age(pd.DataFrame({'age':[np.nan]}),20)['age'].iloc[0]==20


**Complexity**

```text
Time: O(n); Space: O(n)
```

**Common mistake:** Computing the median from validation or test data.


### Exercise — 🟢 Easy: 6. drop_exact_duplicates

        **Goal:** Remove duplicate rows.

        **Example**

        ```text
        Input x=[1,1]
Output: one row with x=1
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def drop_exact_duplicates(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def drop_exact_duplicates(df):
    return df.drop_duplicates().reset_index(drop=True)

assert len(drop_exact_duplicates(pd.DataFrame({'x':[1,1]})))==1


**Complexity**

```text
Time: O(n) average; Space: O(n)
```

**Common mistake:** Mutating the caller when a returned clean frame is expected.


### Exercise — 🟢 Easy: 7. label_distribution

        **Goal:** Return normalized label frequencies.

        **Example**

        ```text
        Input labels=[1,1,0]
Output: class 1 proportion=2/3
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def label_distribution(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def label_distribution(df):
    return df['label'].value_counts(normalize=True)

assert np.isclose(label_distribution(pd.DataFrame({'label':[1,1,0]})).loc[1],2/3)


**Complexity**

```text
Time: O(n) average; Space: O(k)
```

**Common mistake:** Forgetting `normalize=True`.


### Exercise — 🟢 Easy: 8. country_salary

        **Goal:** Mean salary by country.

        **Example**

        ```text
        Input: a/1, a/3
Output: a mean salary=2
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def country_salary(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def country_salary(df):
    return df.groupby('country')['salary'].mean()

assert country_salary(pd.DataFrame({'country':['a','a'],'salary':[1,3]})).loc['a']==2


**Complexity**

```text
Time: O(n) average; Space: O(k)
```

**Common mistake:** Aggregating the wrong numeric column.


### Exercise — 🟠 Medium: 9. country_summary

        **Goal:** Return mean salary and row count.

        **Example**

        ```text
        Input: one country=a salary=2
Output: mean_salary=2, rows=1
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def country_summary(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def country_summary(df):
    return df.groupby('country').agg(mean_salary=('salary','mean'), rows=('salary','size'))

assert country_summary(pd.DataFrame({'country':['a'],'salary':[2]})).loc['a','rows']==1


**Complexity**

```text
Time: O(n) average; Space: O(k)
```

**Common mistake:** Using `count` when missing salaries must still count as rows; use `size`.


### Exercise — 🟠 Medium: 10. left_join_predictions

        **Goal:** Left join predictions on user_id.

        **Example**

        ```text
        Input users 1,2 and prediction for 1
Output: two rows; user 2 score is missing
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def left_join_predictions(df, predictions):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def left_join_predictions(df, predictions):
    return df.merge(predictions,on='user_id',how='left',validate='many_to_one')

assert len(left_join_predictions(pd.DataFrame({'user_id':[1,2]}),pd.DataFrame({'user_id':[1],'score':[.9]})))==2


**Complexity**

```text
Time: O(n+m) average; Space: O(n+m)
```

**Common mistake:** Allowing duplicate right keys to multiply rows silently.


### Exercise — 🟠 Medium: 11. rank_scores

        **Goal:** Sort descending by score.

        **Example**

        ```text
        Input scores=[0.1,0.9]
Output first score=0.9
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def rank_scores(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def rank_scores(df):
    return df.sort_values('score',ascending=False).reset_index(drop=True)

assert rank_scores(pd.DataFrame({'score':[.1,.9]}))['score'].iloc[0]==.9


**Complexity**

```text
Time: O(n log n); Space: O(n)
```

**Common mistake:** Leaving `ascending=True`.


### Exercise — 🟠 Medium: 12. add_margin

        **Goal:** Create margin = top1 - top2.

        **Example**

        ```text
        Input top1=0.9, top2=0.7
Output margin=0.2
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def add_margin(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def add_margin(df):
    return df.assign(margin=df['top1']-df['top2'])

assert add_margin(pd.DataFrame({'top1':[.9],'top2':[.7]}))['margin'].iloc[0] > .19


**Complexity**

```text
Time: O(n); Space: O(n)
```

**Common mistake:** Subtracting in the wrong direction.


### Exercise — 🟠 Medium: 13. parse_month

        **Goal:** Parse date and add month.

        **Example**

        ```text
        Input date=2026-02-01
Output month=2
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def parse_month(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def parse_month(df):
    return df.assign(month=pd.to_datetime(df['date']).dt.month)

assert parse_month(pd.DataFrame({'date':['2026-02-01']}))['month'].iloc[0]==2


**Complexity**

```text
Time: O(n); Space: O(n)
```

**Common mistake:** Using `.dt` before converting to datetime.


### Exercise — 🟠 Medium: 14. unique_users

        **Goal:** Keep latest row per user by date.

        **Example**

        ```text
        Input two dated rows for user 1
Output: latest row only
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def unique_users(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def unique_users(df):
    return df.assign(date=pd.to_datetime(df['date'])).sort_values('date').drop_duplicates('user_id',keep='last')

assert len(unique_users(pd.DataFrame({'user_id':[1,1],'date':['2026-01-01','2026-02-01']})))==1


**Complexity**

```text
Time: O(n log n); Space: O(n)
```

**Common mistake:** Dropping duplicates before sorting chronologically.


### Exercise — 🟠 Medium: 15. class_counts

        **Goal:** Count labels including missing.

        **Example**

        ```text
        Input labels=[1,NaN]
Output: total count=2 including missing
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def class_counts(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def class_counts(df):
    return df['label'].value_counts(dropna=False)

assert class_counts(pd.DataFrame({'label':[1,np.nan]})).sum()==2


**Complexity**

```text
Time: O(n) average; Space: O(k)
```

**Common mistake:** Dropping missing labels unintentionally.


### Exercise — 🟠 Medium: 16. rename_target

        **Goal:** Rename label to target without mutating input.

        **Example**

        ```text
        Input column label
Output column target
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def rename_target(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def rename_target(df):
    return df.rename(columns={'label':'target'})

assert 'target' in rename_target(pd.DataFrame({'label':[1]}))


**Complexity**

```text
Time: O(c); Space: O(nc) for the returned frame
```

**Common mistake:** Using `inplace=True` and unexpectedly mutating shared data.


### Exercise — 🟠 Medium: 17. query_ids

        **Goal:** Return IDs whose score meets threshold.

        **Example**

        ```text
        Input user 1 score=0.9, threshold=0.8
Output: [1]
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def query_ids(df, threshold):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def query_ids(df, threshold):
    return df.loc[df['score']>=threshold,'user_id'].tolist()

assert query_ids(pd.DataFrame({'user_id':[1],'score':[.9]}),.8)==[1]


**Complexity**

```text
Time: O(n); Space: O(n) output
```

**Common mistake:** Returning the Series index instead of `user_id` values.


### Exercise — 🟠 Medium: 18. pivot_metrics

        **Goal:** Pivot mean score by model and split.

        **Example**

        ```text
        Input model=a, split=test, score=0.5
Output: pivot cell (a,test)=0.5
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def pivot_metrics(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def pivot_metrics(df):
    return df.pivot_table(index='model',columns='split',values='score',aggfunc='mean')

assert pivot_metrics(pd.DataFrame({'model':['a'],'split':['test'],'score':[.5]})).loc['a','test']==.5


**Complexity**

```text
Time: O(n) average; Space: O(models × splits)
```

**Common mistake:** Assuming every model/split pair is unique without an aggregation.


### Exercise — 🟠 Medium: 19. string_normalize

        **Goal:** Strip and lowercase a text column.

        **Example**

        ```text
        Input text=' A '
Output text='a'
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def string_normalize(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def string_normalize(df):
    return df.assign(text=df['text'].str.strip().str.lower())

assert string_normalize(pd.DataFrame({'text':[' A ']}))['text'].iloc[0]=='a'


**Complexity**

```text
Time: O(total characters); Space: O(total characters)
```

**Common mistake:** Calling Python string methods directly on a Series.


### Exercise — 🟠 Medium: 20. safe_export

        **Goal:** Return CSV text without the index.

        **Example**

        ```text
        Input one-column DataFrame
Output: CSV text without an index column
        ```

        <details><summary>Hint 1</summary>Write the row mask or column operation first.</details>
        <details><summary>Hint 2</summary>Return a new object when practical.</details>


In [ ]:
def safe_export(df):
    # TODO: Write your pandas solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def safe_export(df):
    return df.to_csv(index=False)

assert not safe_export(pd.DataFrame({'x':[1]})).startswith(',')


**Complexity**

```text
Time: O(nc); Space: O(output CSV)
```

**Common mistake:** Writing the DataFrame index as an unintended first column.


## ML mini-project

Complete before revealing: inspect shape; count missing; remove duplicates; fill age with the **training** median; inspect target distribution; filter SG users; aggregate salary; join predictions; create age bucket; produce CSV text.


In [ ]:
# TODO: build `clean_ml_data` from `df` and `predictions`
clean_ml_data = None


### Mini-project solution


In [ ]:
print("shape", df.shape)
print("missing", df.isna().sum().to_dict())
clean_ml_data = df.drop_duplicates().copy()
training_median = clean_ml_data["age"].median()
clean_ml_data["age"] = clean_ml_data["age"].fillna(training_median)
print("target", clean_ml_data["label"].value_counts(normalize=True).to_dict())
print("SG", clean_ml_data.loc[clean_ml_data["country"].eq("SG")])
print(clean_ml_data.groupby("country")["salary"].mean())
clean_ml_data = clean_ml_data.merge(predictions, on="user_id", how="left", validate="one_to_one")
clean_ml_data["age_bucket"] = pd.cut(clean_ml_data["age"], bins=[0,29,39,np.inf], labels=["20s","30s","40+"])
csv_text = clean_ml_data.to_csv(index=False)
assert not clean_ml_data["age"].isna().any() and csv_text.startswith("user_id")


## Key takeaways

    - Inspect before transforming.
- Fit preprocessing only on training data.
- Validate merge cardinality.

    ## Mistakes to avoid

    - Missing parentheses in filters.
- Chained assignment.
- Silent many-to-many row explosion.

    ## 10 rapid-fire interview questions

    1. **Series vs DataFrame selection syntax?** <details><summary>Answer</summary>One column name usually yields Series; a list of names yields DataFrame.</details>
2. **loc vs iloc?** <details><summary>Answer</summary>Labels vs integer positions.</details>
3. **Is a loc label slice endpoint inclusive?** <details><summary>Answer</summary>Yes.</details>
4. **Is an iloc slice endpoint inclusive?** <details><summary>Answer</summary>No.</details>
5. **Why parentheses around pandas conditions?** <details><summary>Answer</summary>Operator precedence with `&` and `|`.</details>
6. **How inspect missing counts?** <details><summary>Answer</summary>`df.isna().sum()`.</details>
7. **What is preprocessing leakage?** <details><summary>Answer</summary>Using validation/test information to fit transformations.</details>
8. **Where should imputation statistics be fitted?** <details><summary>Answer</summary>Training data only.</details>
9. **What is split-apply-combine?** <details><summary>Answer</summary>Group rows, aggregate/transform groups, combine results.</details>
10. **What is a left merge?** <details><summary>Answer</summary>All left rows plus matching right columns.</details>

    ## Readiness checklist

    - [ ] I can explain loc vs iloc.
- [ ] I can group and merge.
- [ ] I can prevent preprocessing leakage.
